# GPT From Scratch Chatbot (MLX, Apple Silicon)

A small decoder-only Transformer (self-attention, about 11M parameters) trained from scratch, English only.

**How it works:** Run All does this, top to bottom:

1. **First training** (section 7): `make_tokens(TRAIN_FILES)` + `train_model(...)` on one big starting file
   (FineWeb-Edu). Already-trained shards are skipped, so once it is done this step takes seconds.
2. Chat with the model, then **check its accuracy** on fixed test data.
3. **Retrain** (section 10): loads the same model and trains the **next unfinished file** of `RETRAIN_FILES`
   (one file per run, in list order), then checks accuracy again. Run All again for the next file.

`make_tokens` detects each file's kind (txt/csv chat, Wikipedia XML, chat parquet, plain-text parquet), cleans it
and saves tokens as ~5M-token shards in `data/shards/`. Training always continues from `CHECKPOINT`. If you stop
the kernel, it saves; Run All continues from the same spot.


## 1. Settings

In [ ]:
# Files to train, from top to bottom. Mix any kinds; the code detects each file's kind:
#   .txt / .csv       small chat files ("<user>: ..." lines, "<user> ... <end>" blocks, or instruction/response CSV).
#                     All of them are combined into one set and trained TXT_EPOCHS times.
#   enwiki-*.xml-*    Wikipedia dump parts (wikitext is cleaned into plain English text)
#   *.parquet         chat logs ("conversation" or "messages" column, e.g. lmsys / SmolTalk)
#                     or plain text ("text" column, e.g. TinyStories / FineWeb-Edu)
#   other .txt        plain prose without <user> tags (documents separated by blank lines)
#   *.jsonl           chat: ChatML "text" (OpenAssistant), instruction/context/response (Dolly), or "messages"
#   SQuAD-style parquet (context/question/answers): reading questions; unanswerable ones teach the model to
#                     reply "i don't have proper information ..." instead of guessing
# TRAIN_FILES: the one big file a model starts with (section 7). FineWeb-Edu is clean, well-written educational
# English, the best single foundation for grammar and general knowledge (~850M tokens, about 12 h; interrupt any
# time, Run All continues). Everything else is in RETRAIN_FILES (section 10), trained one file per run.
TRAIN_FILES = [
    "chat_dataset/grammar/fineweb_edu/fineweb-edu-10BT-000_00000.parquet",
]
# The second list (RETRAIN_FILES: all other files, in training order) is in the "Retrain" cell at the bottom.

TXT_EPOCHS = 3             # passes over the small txt/csv chat set (10 made model_001 memorise it)
SHARD_EPOCHS = 1           # passes over each wiki/parquet shard
SHARD_TOKENS = 5_000_000   # big files are cut into shards of this many tokens
WIKI_AS_CHAT = True        # also teach "tell me about <title>" -> article intro, so wiki knowledge is usable in chat
TRAINING_ROUND = 1         # shards trained in this round are skipped. Set 2 (3, 4 ...) to train everything again.
START_AT_SHARD = {}        # optional: {"file name": shard number} skips that file's shards before this number in the
                           # next run, e.g. {"fineweb-edu-10BT-001_00000.parquet": 80}. Skipped shards are NOT marked as
                           # trained. Normally not needed: finished shards are always skipped, and an interrupted shard
                           # resumes at its saved step by itself.
ONE_FILE_PER_RUN = True    # Retrain cell: True = each run trains only the next unfinished entry of RETRAIN_FILES (one
                           # file, or a ( ... ) group trained together), then stops. False = the whole list in one run.
MIX_FILES = True           # only when one run trains several files: interleave their shards (spread evenly), so one big
                           # file can't overwrite what the model learned from the others. False = file by file.

# --- model / checkpoint -----------------------------------------------------------------------
# CHECKPOINT = "checkpoints/model_001"   # the first, small model (11M parameters, 256-token context)
CHECKPOINT = "checkpoints/model_002"     # the grown model (51M). Always continues from this checkpoint when it exists.
                                         # A new name starts a brand-new model (or a grown one, see GROW_FROM).
TOKENIZER_FILE = "tokenizer.json"        # English-only (ASCII) byte-level BPE
SHARD_DIR = "data/shards"                # tokens are saved here, one folder per source file

# --- training ---------------------------------------------------------------------------------
BATCH_SIZE = 16
LEARNING_RATE = 1e-4   # 1e-4 for every file: smaller steps, so one file can't swing the model's style as much.
                       # None = auto (3e-4 wiki/text, 1e-4 chat): only for a brand-new model's first big file.
WARMUP_STEPS = 100
DROPOUT = 0.1
GRAD_CLIP = 1.0
MAX_STEPS = None       # e.g. 20 for a quick smoke test (per train_model call); None = train everything
LOG_EVERY = 100
SAVE_EVERY = 500
SEED = 1001

# Growing (section 6b): if CHECKPOINT does not exist yet, it is created as a bigger copy of GROW_FROM that keeps
# everything GROW_FROM learned. Set GROW_FROM = None to start a brand-new model from NEW_MODEL instead.
GROW_FROM = "checkpoints/model_001_best"
GROW_TO_SIZE = {"n_layer": 12, "n_embd": 512, "block_size": 512}   # 2x deeper, 2x wider, 2x context: ~51M parameters

# Only used when no checkpoint exists yet and GROW_FROM is None (brand-new model).
NEW_MODEL = {"block_size": 256, "n_embd": 256, "n_head": 8, "n_layer": 6}

## 2. Imports and paths

In [2]:
import csv
import hashlib
import html
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import unicodedata
from functools import partial
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from tokenizers import Tokenizer

try:
    import mlx.core as mx
    import mlx.nn as nn
    import mlx.optimizers as optim
    from mlx.utils import tree_flatten, tree_unflatten
except ImportError as exc:
    raise ImportError(
        f"MLX is required, but this kernel ({sys.executable}) cannot import it. "
        "In VS Code select the kernel 'GPT MLX Python 3.13' and restart."
    ) from exc

if platform.system() != "Darwin" or platform.machine() != "arm64":
    raise RuntimeError("MLX training needs an Apple-Silicon Mac.")

BASE_DIR = Path.cwd() if (Path.cwd() / "chat_dataset").exists() else Path("/Users/abhishek/gpt_from_scratch_chatbot")


def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else BASE_DIR / path


TOKENIZER_PATH = project_path(TOKENIZER_FILE)
CHECKPOINT_BASE = project_path(CHECKPOINT)
SHARD_ROOT = project_path(SHARD_DIR)

raw_tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
VOCAB_SIZE = raw_tokenizer.get_vocab_size()
PAD_ID, UNK_ID = raw_tokenizer.token_to_id("<pad>"), raw_tokenizer.token_to_id("<unk>")
USER_ID, ASSISTANT_ID = raw_tokenizer.token_to_id("<user>"), raw_tokenizer.token_to_id("<assistant>")
EOS_ID = raw_tokenizer.token_to_id("<eos>")
if None in (PAD_ID, UNK_ID, USER_ID, ASSISTANT_ID, EOS_ID):
    raise RuntimeError(f"{TOKENIZER_PATH} is missing one of <pad> <unk> <user> <assistant> <eos>")
TOKEN_DTYPE = np.uint16 if VOCAB_SIZE <= 65_535 else np.uint32
TOKENIZER_SHA = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()[:16]

mx.random.seed(SEED)
print("MLX", mx.__version__, "|", mx.default_device(), "| vocab", VOCAB_SIZE, "| base", BASE_DIR)

MLX 0.31.2 | Device(gpu, 0) | vocab 12474 | base /Users/abhishek/gpt_from_scratch_chatbot


## 3. Text cleaning and data readers

Everything goes through `to_english_ascii`: accents are folded (`café` → `cafe`), smart quotes and dashes are
replaced, and text that is mostly non-Latin script is dropped as non-English.
Conversations are formatted as `<user>: ...\n<assistant>: ...<eos>\n`. The `<eos>` token teaches the model where to stop.

In [3]:
PUNCTUATION = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'",
    "“": '"', "”": '"', "„": '"', "″": '"',
    "‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-", "―": "-", "−": "-",
    "…": "...", " ": " ", " ": " ", "​": "", "\t": " ", "\r": "",
})


# UTF-8 text that was wrongly decoded as Windows-1252 ("mojibake"), e.g. TinyStories: â€œhiâ€ -> "hi"
MOJIBAKE = [("\u00e2\u20ac\u0153", '"'), ("\u00e2\u20ac\u009d", '"'), ("\u00e2\u20ac\u2122", "'"),
            ("\u00e2\u20ac\u02dc", "'"), ("\u00e2\u20ac\u201c", "-"), ("\u00e2\u20ac\u201d", "-"),
            ("\u00e2\u20ac\u00a6", "..."), ("\u00e2\u20ac", '"')]


def fix_mojibake(text):
    if "\u00e2\u20ac" not in text and "\u00c3" not in text:
        return text
    try:
        return text.encode("cp1252").decode("utf-8")  # exact repair when the whole string allows it
    except (UnicodeEncodeError, UnicodeDecodeError):
        for broken, fixed in MOJIBAKE:
            text = text.replace(broken, fixed)
        return text


def to_english_ascii(text, max_foreign_letters=0.05):
    """Return lowercase-able ASCII English text, or "" if the text is mostly another language."""
    text = unicodedata.normalize("NFKC", fix_mojibake(str(text or ""))).translate(PUNCTUATION)
    text = unicodedata.normalize("NFKD", text)  # é -> e + accent mark (the mark is dropped below)
    letters = foreign = 0
    for char in text:
        if char.isalpha():
            letters += 1
            foreign += ord(char) > 127
    if letters == 0 or foreign / letters > max_foreign_letters:
        return ""
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^\x20-\x7e\n]", "", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def format_conversation(turns):
    """turns: list of (role, text) with role "user" / "assistant". Returns "" if unusable."""
    parts = []
    has_reply = False
    for role, text in turns:
        text = to_english_ascii(text)
        if not text:
            return ""  # one non-English or empty turn makes the conversation incoherent, so drop all of it
        if role == "user":
            parts.append(f"<user>: {text}\n")
        else:
            parts.append(f"<assistant>: {text}<eos>\n")
            has_reply = True
    return "".join(parts) if has_reply else ""


# ---------- txt / csv chat files ----------
ROLE_LINE = re.compile(r"^\s*<\|?(user|assistant|end)\|?>\s*:?\s?", re.IGNORECASE)


def parse_chat_txt(path):
    conversations, turns, role, lines = [], [], None, []
    previous_blank = False

    def close_turn():
        nonlocal role, lines
        text = "\n".join(lines).strip()
        if role and text:
            turns.append((role, text))
        role, lines = None, []

    def close_conversation():
        nonlocal turns
        close_turn()
        if turns:
            conversations.append(turns)
        turns = []

    for line in Path(path).read_text(encoding="utf-8", errors="replace").splitlines():
        match = ROLE_LINE.match(line)
        if match:
            tag = match.group(1).lower()
            close_turn()
            if tag == "end":
                close_conversation()
            else:
                # A blank line before a new <user> turn starts a new conversation.
                if tag == "user" and previous_blank:
                    close_conversation()
                role, lines = tag, [line[match.end():]]
        elif role:
            lines.append(line)
        previous_blank = not line.strip()
    close_conversation()
    return conversations


def parse_chat_csv(path):
    conversations = []
    with Path(path).open(encoding="utf-8", errors="replace", newline="") as handle:
        for row in csv.DictReader(handle):
            question, answer = row.get("instruction"), row.get("response")
            if question and answer:
                # Bitext uses {{Order Number}} placeholders; keep them readable as [order number].
                fill = lambda s: re.sub(r"\{\{\s*([^}]+?)\s*\}\}", r"[\1]", s)
                conversations.append([("user", fill(question)), ("assistant", fill(answer))])
    return conversations


# ---------- parquet chat logs (lmsys-chat style) ----------
PARQUET_ROLES = {"user": "user", "human": "user", "assistant": "assistant", "gpt": "assistant", "bot": "assistant"}


NO_INFO_REPLY = "i don't have proper information about that in the given text."
READING_PROMPT = "read the text and answer the question. if the answer is not in the text, say so."
ENGLISH_WORDS = set("""the a an and or but of to in on at for with from by is are was were be been it this that these
those i you he she we they my your his her our their not no yes do does did have has had can could will would should
what which who when where why how if then than there here about as so very just also more most some any all one""".split())


def looks_english(text, min_ratio=0.2):
    """Rough check for English in Latin-script text (to_english_ascii only catches other alphabets)."""
    words = re.findall(r"[a-z']+", str(text).lower())
    if len(words) < 6:
        return True
    return sum(w in ENGLISH_WORDS for w in words) / len(words) >= min_ratio


def reading_prompt(row):
    return f"{READING_PROMPT}\n\n{row['context']}\n\nquestion: {row['question']}"


def squad_row_to_text(row):
    """SQuAD v2: answer from the text, or say the information is not there (about a third are unanswerable)."""
    answers = (row.get("answers") or {}).get("text") or []
    answer = answers[0].strip().rstrip(".") + "." if answers else NO_INFO_REPLY
    return format_conversation([("user", reading_prompt(row)), ("assistant", answer)])


def parquet_row_to_text(row):
    if "question" in row and "context" in row:
        return squad_row_to_text(row)
    if bool(row.get("redacted")):
        return ""
    if "language" in row and str(row.get("language") or "").strip().lower() not in ("english", "en"):
        return ""
    if any(isinstance(item, dict) and item.get("flagged") for item in (row.get("openai_moderation") or [])):
        return ""
    turns, system = [], ""
    for message in row.get("conversation") or row.get("messages") or []:  # lmsys / SmolTalk
        raw_role = str((message or {}).get("role", "")).strip().lower()
        if raw_role == "system":
            system = str(message.get("content") or "").strip()  # an instruction like "rewrite this text"
            continue
        role = PARQUET_ROLES.get(raw_role)
        if role:
            content = message.get("content")
            if role == "user" and system and not turns:
                content = f"{system}\n\n{content}"  # keep the instruction, or the reply makes no sense
                system = ""
            turns.append((role, content))
    return format_conversation(turns)


def iter_parquet_rows(path, start_row, columns, batch_size=256):
    """Yields (row number, row) from start_row on. Whole row groups before start_row are skipped without being
    read, so resuming in the middle of a big file is instant."""
    parquet_file = pq.ParquetFile(path)
    columns = [c for c in columns if c in parquet_file.schema_arrow.names]
    row_number = 0
    for group in range(parquet_file.metadata.num_row_groups):
        group_rows = parquet_file.metadata.row_group(group).num_rows
        if row_number + group_rows <= start_row:
            row_number += group_rows
            continue
        for batch in parquet_file.iter_batches(batch_size=batch_size, row_groups=[group], columns=columns, use_threads=False):
            for row in batch.to_pylist():
                if row_number >= start_row:
                    yield row_number, row
                row_number += 1


def iter_parquet(path, start_row):
    """Chat parquet. Yields (next_row_position, text). Position = rows consumed so far, used to resume."""
    columns = ("conversation", "messages", "language", "redacted", "openai_moderation", "context", "question", "answers")
    for row_number, row in iter_parquet_rows(path, start_row, columns):
        yield row_number + 1, parquet_row_to_text(row)


# ---------- Wikipedia XML dump ----------
END_SECTIONS = {"references", "external links", "see also", "further reading", "notes", "bibliography",
                "sources", "citations", "footnotes", "notes and references", "works cited", "explanatory notes"}
DROP_LINK_PREFIXES = {"file", "image", "category", "media", "wikipedia", "wp", "wikt", "wiktionary", "help",
                      "template", "portal", "special", "user", "talk", "commons", "s", "q", "d", "w", "meta"}


def iter_wiki_pages(path, start_offset):
    """Streams <page> blocks. Yields (byte offset after the page, page xml). Offsets let us resume instantly."""
    with open(path, "rb") as handle:
        handle.seek(start_offset)
        page = None
        while True:
            line = handle.readline()
            if not line:
                return
            stripped = line.strip()
            if stripped == b"<page>":
                page = []
            elif page is not None:
                if stripped == b"</page>":
                    yield handle.tell(), b"".join(page).decode("utf-8", "replace")
                    page = None
                else:
                    page.append(line)


def _link(match):
    inner = match.group(1)
    target = inner.split("|", 1)[0]
    if ":" in target:
        prefix, rest = target.split(":", 1)
        prefix = prefix.strip().lower()
        if prefix in DROP_LINK_PREFIXES or (re.fullmatch(r"[a-z]{2,3}(-[a-z]+)?", prefix) and not rest.startswith(" ")):
            return ""  # files, categories, interlanguage links like [[fr:...]]
    return inner.split("|")[-1]


def clean_wikitext(text):
    text = html.unescape(text)
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"<ref[^>]*/>", "", text, flags=re.I)
    text = re.sub(r"<ref[^>]*>.*?</ref>", "", text, flags=re.S | re.I)
    text = re.sub(r"<(gallery|math|chem|score|timeline|syntaxhighlight|source|pre|code|imagemap|mapframe|graph|table)[^>]*>.*?</\1>",
                  "", text, flags=re.S | re.I)
    for _ in range(12):  # templates / tables nest, so remove the innermost ones repeatedly
        new = re.sub(r"\{\{[^{}]*\}\}", "", text)
        new = re.sub(r"\{\|[^{}]*?\|\}", "", new, flags=re.S)
        if new == text:
            break
        text = new
    for _ in range(6):
        new = re.sub(r"\[\[([^\[\]]*)\]\]", _link, text)
        if new == text:
            break
        text = new
    text = re.sub(r"\[https?://[^\s\]]+\s*([^\]]*)\]", r"\1", text)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"'{2,}", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"__[A-Z]+__", "", text)
    text = html.unescape(text)

    lines = []
    for line in text.split("\n"):
        line = line.strip()
        heading = re.fullmatch(r"(=+)\s*(.*?)\s*=+", line)
        if heading:
            if heading.group(2).strip().lower() in END_SECTIONS:
                break
            lines.append("")
            lines.append(heading.group(2).strip())
            continue
        if not line or line[0] in "{}|!" or line.startswith(("[[", "]]")):
            lines.append("")
            continue
        line = re.sub(r"^[*#:;]+\s*", "", line)
        line = re.sub(r"\(\s*[,;]?\s*\)", "", line)
        line = re.sub(r"\s+([,.;:])", r"\1", line)
        lines.append(line)
    text = "\n".join(lines)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def wiki_page_to_texts(page_xml):
    title = re.search(r"<title>(.*?)</title>", page_xml, flags=re.S)
    namespace = re.search(r"<ns>(\d+)</ns>", page_xml)
    body = re.search(r"<text[^>]*>(.*?)</text>", page_xml, flags=re.S)
    if not title or not body or (namespace and namespace.group(1) != "0") or "<redirect" in page_xml:
        return []
    title = html.unescape(title.group(1)).strip()
    if "(disambiguation)" in title or title.lower().startswith("list of"):
        return []
    if re.match(r"\s*#redirect", body.group(1), flags=re.I):
        return []
    article = to_english_ascii(clean_wikitext(body.group(1)))
    article = re.sub(r"\(\s*[,;:]*\s*\)", "", article)              # "()" left by removed templates
    article = re.sub(r"\(\s*[,;:]+\s*", "(", article)                # "(,;: alga )" -> "(alga )"
    article = re.sub(r" +([,.;:)])", r"\1", article)
    article = keep_complete_paragraphs(article)
    title = to_english_ascii(title)
    if not title or len(article) < 300:
        return []
    texts = [f"{title}\n\n{article}<eos>\n"]
    if WIKI_AS_CHAT:
        intro = article.split("\n\n", 1)[0]
        sentences = re.split(r"(?<=[.!?])\s+", intro)
        answer = " ".join(sentences[:3]).strip()
        if 60 <= len(answer) <= 700:
            subject = re.sub(r"\s*\(.*?\)\s*$", "", title)
            question = random.Random(title).choice([
                f"tell me about {subject}.", f"what do you know about {subject}?",
                f"can you explain {subject}?", f"who or what is {subject}?",
            ])
            texts.append(format_conversation([("user", question), ("assistant", answer)]))
    return texts


def iter_wiki(path, start_offset):
    for offset, page_xml in iter_wiki_pages(path, start_offset):
        yield offset, "".join(wiki_page_to_texts(page_xml))


def keep_complete_paragraphs(text, min_words=6):
    """Drops headings, list fragments and broken lines: keeps paragraphs that read like real sentences."""
    kept = []
    for paragraph in text.split("\n\n"):
        lines = [line for line in paragraph.split("\n")
                 if len(line.split()) >= min_words and line.rstrip()[-1:] in ".!?\"'"]
        if lines:
            kept.append("\n".join(lines))
    return "\n\n".join(kept)


# ---------- plain prose: parquet with a "text" column (TinyStories, FineWeb-Edu ...) or prose .txt ----------
def iter_text_parquet(path, start_row):
    """Plain-text parquet. Yields (next_row_position, text). Each row is one document, ended with <eos>."""
    for row_number, row in iter_parquet_rows(path, start_row, ("text", "language"), batch_size=512):
        language = str(row.get("language") or "en").lower()
        text = to_english_ascii(row.get("text")) if language.startswith("en") else ""
        yield row_number + 1, f"{text}<eos>\n" if len(text) >= 50 else ""


def iter_prose_txt(path, start_offset):
    """Plain prose .txt: documents are separated by blank lines or <|endoftext|>. Yields (byte offset, text)."""
    with open(path, "rb") as handle:
        handle.seek(start_offset)
        lines = []
        while True:
            line = handle.readline()
            text = line.decode("utf-8", "replace").strip()
            if not line or not text or text == "<|endoftext|>":
                document = to_english_ascii(" ".join(lines))
                if lines:
                    yield handle.tell(), f"{document}<eos>\n" if len(document) >= 50 else ""
                lines = []
                if not line:
                    return
            else:
                lines.append(text)


# ---------- chat .jsonl: OpenAssistant (ChatML text), Dolly (instruction/context/response), messages lists ----------
CHATML_TURN = re.compile(r"<\|im_start\|>\s*(system|user|assistant)\s*\n(.*?)(?:<\|im_end\|>|$)", re.S)


def jsonl_row_to_text(row):
    if isinstance(row.get("text"), str) and "<|im_start|>" in row["text"]:
        turns, system = [], ""
        for role, content in CHATML_TURN.findall(row["text"]):
            if role == "system":
                system = content.strip()
                continue
            if role == "user" and system and not turns:
                content, system = f"{system}\n\n{content}", ""
            turns.append((role, content.strip()))
        if not all(looks_english(content) for _, content in turns):
            return ""  # OpenAssistant is multilingual; keep English conversations only
        return format_conversation(turns)
    question = row.get("instruction") or row.get("prompt") or row.get("question")
    answer = row.get("response") or row.get("output") or row.get("answer")
    if question and answer:
        context = row.get("context") or row.get("input") or ""
        return format_conversation([("user", f"{question}\n\n{context}".strip()), ("assistant", answer)])
    if row.get("messages") or row.get("conversation"):
        return parquet_row_to_text(row)
    return ""


def iter_jsonl(path, start_offset):
    """Yields (byte offset after the line, formatted text); the byte offset makes resuming instant."""
    with open(path, "rb") as handle:
        handle.seek(start_offset)
        while True:
            line = handle.readline()
            if not line:
                return
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                row = {}
            yield handle.tell(), jsonl_row_to_text(row) if isinstance(row, dict) else ""


## 4. `make_tokens(files)`: read any list of files and turn it into tokens

For each file it detects the kind, reads and cleans the text, and tokenizes it. Big files (wiki, parquet) are saved as
`data/shards/<file name>/shard_000000.npy`, `shard_000001.npy`, … with a `progress.json`, so tokenizing never has to be
repeated and can resume if stopped. All txt/csv files in the list are combined into one deduplicated chat set.
It returns the list of shards in training order.

In [4]:
def atomic_write_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def encode_texts(texts):
    ids = []
    for encoding in raw_tokenizer.encode_batch(texts):
        ids.extend(encoding.ids)
    return ids


# One process at a time may tokenize/train, so two runs never write the same checkpoint or shards.
LOCK_PATH = Path(f"{CHECKPOINT_BASE}.lock")


class training_lock:
    def __enter__(self):
        if LOCK_PATH.exists():
            pid = int(LOCK_PATH.read_text().strip() or 0)
            try:
                os.kill(pid, 0)
                alive = pid not in (0, os.getpid())
            except ProcessLookupError:
                alive = False  # stale lock from a crashed run
            except PermissionError:
                alive = True
            if alive:
                raise RuntimeError(f"another training process (pid {pid}) is using {CHECKPOINT_BASE.name}. Wait for it, "
                                   f"or stop it first: run `kill {pid}` in a terminal (it saves and exits), "
                                   f"or interrupt the other notebook kernel.")
        LOCK_PATH.parent.mkdir(parents=True, exist_ok=True)
        LOCK_PATH.write_text(str(os.getpid()))

    def __exit__(self, *exc):
        LOCK_PATH.unlink(missing_ok=True)


def file_kind(path):
    """'chat_text' (small chat .txt/.csv), 'parquet' (chat logs), 'text' (plain prose), or 'wiki' (XML dump)."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return "chat_text"
    if suffix == ".jsonl":
        return "jsonl"
    if suffix == ".txt":
        with open(path, "rb") as handle:
            head = handle.read(65536).decode("utf-8", "replace")
        return "chat_text" if re.search(r"<\|?(user|assistant)\|?>", head) else "text"
    if suffix == ".parquet":
        names = pq.ParquetFile(path).schema_arrow.names
        if "conversation" in names or "messages" in names or {"context", "question", "answers"} <= set(names):
            return "parquet"
        if "text" in names:
            return "text"
        raise ValueError(f"{path.name}: parquet needs a 'conversation', 'messages' or 'text' column, has {names}")
    with open(path, "rb") as handle:
        head = handle.read(4096)
    if b"<mediawiki" in head:
        return "wiki"
    if head[:3] == b"BZh":
        raise ValueError(f"{path.name} is bz2-compressed: decompress it first (bunzip2 -k {path.name})")
    raise ValueError(f"don't know how to read {path.name}: use .txt, .csv, .parquet or a Wikipedia XML dump")


def read_texts(path, kind, start=0):
    """Reads one file of any kind from position `start`. Yields (position after this item, formatted text)."""
    if kind == "wiki":
        yield from iter_wiki(path, start)
    elif kind == "parquet":
        yield from iter_parquet(path, start)
    elif kind == "jsonl":
        yield from iter_jsonl(path, start)
    elif kind == "text":
        yield from (iter_text_parquet if path.suffix.lower() == ".parquet" else iter_prose_txt)(path, start)
    else:
        conversations = parse_chat_csv(path) if path.suffix.lower() == ".csv" else parse_chat_txt(path)
        for index, turns in enumerate(conversations[start:], start + 1):
            yield index, format_conversation(turns)


def load_txt_corpus(paths):
    texts, seen = [], set()
    for path in paths:
        path = project_path(path)
        kept = total = 0
        for _, text in read_texts(path, "chat_text"):
            total += 1
            key = hashlib.md5(text.lower().encode()).digest()
            if text and key not in seen:  # skip exact duplicate conversations across files
                seen.add(key)
                texts.append(text)
                kept += 1
        print(f"  {path.name}: {total:,} conversations, {kept:,} kept")
    return texts


def shard_folder(source_path):
    return SHARD_ROOT / source_path.name


def load_progress(source_path):
    folder = shard_folder(source_path)
    progress_path = folder / "progress.json"
    if progress_path.exists():
        progress = json.loads(progress_path.read_text())
        if progress["tokenizer_sha"] != TOKENIZER_SHA:
            raise RuntimeError(f"{folder} was tokenized with a different tokenizer. Delete that folder to rebuild it.")
        return progress
    return {"source": source_path.name, "source_size": source_path.stat().st_size,
            "tokenizer_sha": TOKENIZER_SHA, "next_position": 0, "finished": False, "shards": []}


def build_next_shard(kind, source_path, progress):
    """Reads the file from where the previous shard stopped until SHARD_TOKENS tokens are collected."""
    folder = shard_folder(source_path)
    folder.mkdir(parents=True, exist_ok=True)
    index = len(progress["shards"])
    start = progress["next_position"]
    tokens, pending, position, documents = [], [], start, 0
    started = time.time()
    for position, text in read_texts(source_path, kind, start):
        if text:
            pending.append(text)
            documents += 1
        if len(pending) >= 64:
            tokens.extend(encode_texts(pending))
            pending = []
            if len(tokens) >= SHARD_TOKENS:
                break
    else:
        progress["finished"] = True  # reached the end of the file
    if pending:
        tokens.extend(encode_texts(pending))

    if tokens:
        name = f"shard_{index:06d}.npy"
        np.save(folder / name, np.asarray(tokens, dtype=TOKEN_DTYPE), allow_pickle=False)
        progress["shards"].append({"file": name, "tokens": len(tokens), "documents": documents,
                                   "start": start, "end": position})
        print(f"    {name}: {len(tokens):,} tokens from {documents:,} documents ({time.time() - started:.0f}s)")
    if progress["finished"] and not progress["shards"]:
        # Nothing usable was read from the whole file: don't record it as done, so a fixed reader can retry it.
        raise RuntimeError(f"no text could be read from {source_path.name} (read as kind '{kind}'). "
                           "Check that the notebook code is up to date (restart the kernel) and the file's columns/format.")
    progress["next_position"] = position
    atomic_write_json(progress, folder / "progress.json")
    return progress


def chat_text_folder(paths):
    """Folder of the combined txt/csv chat set; its name changes when any of the files change."""
    key = "|".join(f"{p.name}:{p.stat().st_size}:{p.stat().st_mtime_ns}" for p in paths)
    return SHARD_ROOT / f"chat_text_{hashlib.md5(key.encode()).hexdigest()[:8]}"


def make_chat_text_shard(paths):
    """All txt/csv chat files become one deduplicated token set (its id changes when the files change)."""
    folder = chat_text_folder(paths)
    path = folder / "shard_000000.npy"
    if not path.exists():
        print("  reading chat text files:")
        tokens = np.asarray(encode_texts(load_txt_corpus(paths)), dtype=TOKEN_DTYPE)
        folder.mkdir(parents=True, exist_ok=True)
        np.save(path, tokens, allow_pickle=False)
        atomic_write_json({"files": [p.name for p in paths], "tokens": len(tokens)}, folder / "progress.json")
    tokens = len(np.load(path, mmap_mode="r"))
    print(f"  chat text ({len(paths)} files): {tokens / 1e6:.2f}M tokens, trained {TXT_EPOCHS} epochs")
    return {"id": f"{folder.name}/{path.name}", "path": path, "kind": "chat_text", "tokens": tokens}


def make_tokens(files):
    """Reads every file in the list (any kind) and makes tokens. Returns the shards in training order."""
    paths = [project_path(f) for f in files]
    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"files not found: {missing}")
    kinds = {path: file_kind(path) for path in paths}
    chat_text_paths = [p for p in paths if kinds[p] == "chat_text"]
    shards = []
    started = time.time()
    with training_lock():
        print(f"making tokens for {len(paths)} files ...")
        for path in paths:
            kind = kinds[path]
            if kind == "chat_text":
                if path == chat_text_paths[0]:  # all chat text files go in at the first one's position
                    shards.append(make_chat_text_shard(chat_text_paths))
                continue
            progress = load_progress(path)
            if not progress["finished"]:
                print(f"  {path.name} ({kind}): tokenizing from shard {len(progress['shards'])} ...")
            while not progress["finished"]:
                progress = build_next_shard(kind, path, progress)
            for record in progress["shards"]:
                shards.append({"id": f"{path.name}/{record['file']}", "path": shard_folder(path) / record["file"],
                               "kind": kind, "tokens": record["tokens"]})
            total = sum(r["tokens"] for r in progress["shards"])
            print(f"  {path.name} ({kind}): {len(progress['shards'])} shards, {total / 1e6:,.1f}M tokens")
    total = sum(s["tokens"] for s in shards)
    print(f"done: {len(shards)} shards, {total / 1e6:,.1f}M tokens ({(time.time() - started) / 60:.1f} min)")
    if MIX_FILES and len({s["id"].split("/")[0] for s in shards}) > 1:
        shards = mix_shards(shards)
        print("training order: shards of all files interleaved (MIX_FILES = True)")
    return shards


def mix_shards(shards):
    """Spreads each file's shards evenly over the whole run (keeping each file's own order), so the training
    alternates between files instead of doing one long block per file."""
    by_file = {}
    for shard in shards:
        by_file.setdefault(shard["id"].split("/")[0], []).append(shard)
    keyed = [((i + 0.5) / len(group), file_index, shard)
             for file_index, group in enumerate(by_file.values()) for i, shard in enumerate(group)]
    return [shard for _, _, shard in sorted(keyed, key=lambda item: item[:2])]


def flatten_files(entries):
    """RETRAIN_FILES entries are file paths or ( ... ) groups of paths; returns one flat list of paths."""
    return [f for entry in entries for f in (entry if isinstance(entry, (list, tuple)) else [entry])]


def next_files_to_train(entries):
    """For ONE_FILE_PER_RUN: the first entry of the list that still has untrained shards. An entry is one file, or a
    ( ... ) group of files trained together in one run (interleaved when MIX_FILES = True). Loose txt/csv chat files
    count as one entry (they are trained together). Returns [] when everything is trained."""
    state_path = Path(f"{CHECKPOINT_BASE}.json")
    trained = json.loads(state_path.read_text())["trained_shards"] if state_path.exists() else {}

    def done(shard_id):
        record = trained.get(shard_id)
        return record is not None and record.get("round", 1) >= TRAINING_ROUND

    missing = [f for f in flatten_files(entries) if not project_path(f).exists()]
    if missing:
        raise FileNotFoundError(f"files not found: {missing}")
    units, loose_chat = [], []
    for entry in entries:
        group = [project_path(f) for f in (entry if isinstance(entry, (list, tuple)) else [entry])]
        if len(group) == 1 and file_kind(group[0]) == "chat_text":
            if not loose_chat:
                units.append(loose_chat)  # filled below; sits at the first loose chat file's position
            loose_chat.append(group[0])
        else:
            units.append(group)

    def complete(group):
        chat = [p for p in group if file_kind(p) == "chat_text"]
        if chat and not done(f"{chat_text_folder(chat).name}/shard_000000.npy"):
            return False
        for p in group:
            if file_kind(p) != "chat_text":
                progress = load_progress(p)
                if not (progress["finished"] and all(done(f"{p.name}/{r['file']}") for r in progress["shards"])):
                    return False
        return True

    status = [complete(group) for group in units]
    for number, (group, is_complete) in enumerate(zip(units, status), 1):
        if not is_complete:
            print(f"entry {number} of {len(units)} in the list ({sum(status)} done): {', '.join(p.name for p in group)}")
            return [str(p) for p in group]
    print(f"all {len(units)} entries in the list are trained (round {TRAINING_ROUND})")
    return []


def assistant_loss_mask(tokens):
    """mask[i] = 1 if token i+1 is part of an assistant reply (including its <eos>)."""
    events = np.full(len(tokens), -1, dtype=np.int8)
    events[tokens == ASSISTANT_ID] = 1
    events[tokens == USER_ID] = 0
    events[tokens == EOS_ID] = 0
    last_event = np.where(events >= 0, np.arange(len(tokens)), 0)
    np.maximum.accumulate(last_event, out=last_event)
    return (events[last_event] == 1).astype(np.float32)

## 5. The model (decoder-only Transformer)

Same architecture and parameter names as before, so existing checkpoints load. Attention uses MLX's fused causal kernel.

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        if n_embd % n_head:
            raise ValueError("n_embd must be divisible by n_head")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.projection = nn.Linear(n_embd, n_embd)
        self.residual_dropout = nn.Dropout(dropout)

    def __call__(self, x):
        batch, length, channels = x.shape
        query, key, value = mx.split(self.qkv(x), 3, axis=-1)
        heads = lambda t: t.reshape(batch, length, self.n_head, self.head_dim).transpose(0, 2, 1, 3)
        output = mx.fast.scaled_dot_product_attention(heads(query), heads(key), heads(value),
                                                      scale=self.scale, mask="causal")
        output = output.transpose(0, 2, 1, 3).reshape(batch, length, channels)
        return self.residual_dropout(self.projection(output))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.up = nn.Linear(n_embd, 4 * n_embd)
        self.activation = nn.GELU()
        self.down = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def __call__(self, x):
        return self.dropout(self.down(self.activation(self.up(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.attention_norm = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention(n_embd, n_head, dropout)
        self.feed_forward_norm = nn.LayerNorm(n_embd)
        self.feed_forward = FeedForward(n_embd, dropout)

    def __call__(self, x):
        x = x + self.attention(self.attention_norm(x))
        return x + self.feed_forward(self.feed_forward_norm(x))


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = [TransformerBlock(n_embd, n_head, dropout) for _ in range(n_layer)]
        self.final_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

    def __call__(self, input_ids):
        length = input_ids.shape[1]
        x = self.token_embedding(input_ids) + self.position_embedding(mx.arange(length))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))


def masked_loss(model, inputs, targets, mask):
    losses = nn.losses.cross_entropy(model(inputs), targets, reduction="none")
    return (losses * mask).sum() / mx.maximum(mask.sum(), 1.0)

## 6. `train_model(shards)`: train on all the tokens

Loads the model from `checkpoints/chatbot_mlx` (or creates a new one), then goes through the shards in order.
Each shard is shuffled into 256-token windows and trained with next-token prediction. Wikipedia is trained on all
tokens, chat data only on the assistant replies. After every shard the checkpoint is saved and the shard is recorded
as trained, so it is skipped next time. Before each shard, the previous checkpoint is copied to `chatbot_mlx.previous.*`.

In [6]:
def checkpoint_paths(base=None):
    base = base or CHECKPOINT_BASE
    return {"weights": Path(f"{base}.safetensors"), "optimizer": Path(f"{base}.optimizer.safetensors"),
            "state": Path(f"{base}.json")}


def load_model(dropout=0.0, with_optimizer=False, base=None):
    paths = checkpoint_paths(project_path(base) if base else CHECKPOINT_BASE)
    if paths["state"].exists():
        state = json.loads(paths["state"].read_text())
    else:
        state = {"model": {**NEW_MODEL, "vocab_size": VOCAB_SIZE}, "global_step": 0, "tokens_seen": 0,
                 "trained_shards": {}, "in_progress": None, "history": []}
    shape = state["model"]
    if shape["vocab_size"] != VOCAB_SIZE:
        raise RuntimeError(f"checkpoint vocab {shape['vocab_size']} != tokenizer vocab {VOCAB_SIZE}")
    model = TinyGPT(VOCAB_SIZE, shape["block_size"], shape["n_embd"], shape["n_head"], shape["n_layer"], dropout)
    if paths["weights"].exists():
        model.load_weights(str(paths["weights"]))
        print(f"loaded {paths['weights'].name} (step {state['global_step']:,}, {state.get('tokens_seen', 0) / 1e6:,.0f}M tokens seen)")
    else:
        print("no checkpoint found: starting a NEW model")
    mx.eval(model.parameters())
    if not with_optimizer:
        model.eval()
        return model, state
    optimizer = optim.AdamW(learning_rate=1e-4, weight_decay=0.01)
    optimizer.init(model.trainable_parameters())
    if paths["optimizer"].exists():
        optimizer.state = tree_unflatten(list(mx.load(str(paths["optimizer"])).items()))
        optimizer.init(model.trainable_parameters())  # fills anything missing, keeps loaded moments
    return model, state, optimizer


def save_checkpoint(model, optimizer, state):
    paths = checkpoint_paths()
    paths["weights"].parent.mkdir(parents=True, exist_ok=True)
    mx.eval(model.parameters(), optimizer.state)
    weights_tmp = paths["weights"].with_name(paths["weights"].stem + ".tmp.safetensors")
    optimizer_tmp = paths["optimizer"].with_name(paths["optimizer"].stem + ".tmp.safetensors")
    model.save_weights(str(weights_tmp))
    mx.save_safetensors(str(optimizer_tmp), dict(tree_flatten(optimizer.state)))
    os.replace(weights_tmp, paths["weights"])
    os.replace(optimizer_tmp, paths["optimizer"])
    atomic_write_json(state, paths["state"])


def backup_checkpoint():
    for path in checkpoint_paths().values():
        if path.exists():
            shutil.copy2(path, path.with_name(path.name.replace(CHECKPOINT_BASE.name, CHECKPOINT_BASE.name + ".previous", 1)))


def evaluate(model, tokens, mask, starts, block_size, max_batches=40):
    model.eval()
    total, count = 0.0, 0
    for i in range(0, min(len(starts), max_batches * BATCH_SIZE), BATCH_SIZE):
        x, y, m = make_batch(tokens, mask, starts[i:i + BATCH_SIZE], block_size)
        total += masked_loss(model, x, y, m).item()
        count += 1
    model.train()
    return total / max(count, 1)


def make_batch(tokens, mask, starts, block_size):
    x = np.stack([tokens[s:s + block_size] for s in starts]).astype(np.int32)
    y = np.stack([tokens[s + 1:s + block_size + 1] for s in starts]).astype(np.int32)
    m = np.stack([mask[s:s + block_size] for s in starts])
    return mx.array(x), mx.array(y), mx.array(m)


def train_model(shards):
    """Trains on the shards in order. Skips shards already trained this TRAINING_ROUND; resumes an interrupted one."""
    with training_lock():
        model, state, optimizer = load_model(dropout=DROPOUT, with_optimizer=True)
        block_size = model.block_size

        def already_trained(shard):
            record = state["trained_shards"].get(shard["id"])
            return record is not None and record.get("round", 1) >= TRAINING_ROUND

        def file_of(shard):
            return shard["id"].split("/")[0]

        def number_of(shard):
            return int(re.search(r"shard_(\d+)", shard["id"]).group(1))

        todo = [shard for shard in shards if not already_trained(shard)]
        start_at = globals().get("START_AT_SHARD") or {}
        skipped = [s for s in todo if number_of(s) < start_at.get(file_of(s), 0)]
        todo = [s for s in todo if s not in skipped]
        in_progress = state.get("in_progress") or {}
        todo.sort(key=lambda shard: shard["id"] != in_progress.get("shard_id"))  # resume an interrupted shard first

        # Per-file summary, so it is clear where training continues (shard numbers are the file's own numbers).
        per_file = {}
        for shard in shards:
            per_file.setdefault(file_of(shard), []).append(shard)
        print(f"{len(shards)} shards in the list: {len(shards) - len(todo) - len(skipped)} already trained, "
              f"{len(todo)} to train ({sum(s['tokens'] for s in todo) / 1e6:,.1f}M tokens)"
              + (f", {len(skipped)} skipped by START_AT_SHARD" if skipped else ""))
        for name, group in per_file.items():
            done = sum(already_trained(s) for s in group)
            print(f"  {name}: {done} of {len(group)} shards trained")
        if in_progress.get("shard_id") in {s["id"] for s in todo}:
            print(f"  continuing {in_progress['shard_id']} at batch {in_progress['batch_index']:,} (saved position)")

        loss_and_grad = nn.value_and_grad(model, masked_loss)
        compiled_state = [model.state, optimizer.state, mx.random.state]

        @partial(mx.compile, inputs=compiled_state, outputs=compiled_state)
        def train_step(x, y, m):
            loss, grads = loss_and_grad(model, x, y, m)
            grads, grad_norm = optim.clip_grad_norm(grads, GRAD_CLIP)
            optimizer.update(model, grads)
            return loss, grad_norm

        steps_this_call = 0
        call_started = time.time()
        for number, shard in enumerate(todo, 1):
            shard_id, kind = shard["id"], shard["kind"]
            tokens = np.load(shard["path"], allow_pickle=False)
            chat = kind in ("chat_text", "parquet", "jsonl")
            epochs = TXT_EPOCHS if kind == "chat_text" else SHARD_EPOCHS
            learning_rate = LEARNING_RATE or (1e-4 if chat else 3e-4)
            if len(tokens) < (BATCH_SIZE + 2) * (block_size + 1):
                print(f"skipping {shard_id}: only {len(tokens):,} tokens")
                continue
            mask = assistant_loss_mask(tokens) if chat else np.ones(len(tokens), dtype=np.float32)

            # Non-overlapping windows; every 50th window is held out to measure validation loss.
            all_starts = np.arange(0, len(tokens) - block_size - 1, block_size)
            val_starts = all_starts[::50]
            train_starts = np.setdiff1d(all_starts, val_starts)
            batches_per_epoch = len(train_starts) // BATCH_SIZE
            total_steps = batches_per_epoch * epochs

            progress = state.get("in_progress") or {}
            epoch, batch_index = (progress["epoch"], progress["batch_index"]) if progress.get("shard_id") == shard_id else (0, 0)
            run_seed = int(hashlib.md5(shard_id.encode()).hexdigest()[:8], 16) ^ SEED
            group = per_file[file_of(shard)]
            done_in_file = sum(already_trained(s) for s in group)
            print(f"\n=== this run {number}/{len(todo)} | {file_of(shard)} shard {number_of(shard)} "
                  f"({done_in_file} of {len(group)} done) ({kind}) | {len(tokens):,} tokens | "
                  f"{total_steps:,} steps | lr {learning_rate:g} | loss on {'assistant replies' if chat else 'all tokens'}"
                  + (f" | resuming at step {epoch * batches_per_epoch + batch_index:,}" if epoch or batch_index else ""))

            def lr_at(step):
                if step < WARMUP_STEPS:
                    return learning_rate * (step + 1) / WARMUP_STEPS
                ratio = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
                return learning_rate * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, ratio))))

            def checkpoint(done):
                state["in_progress"] = None if done else {"shard_id": shard_id, "epoch": epoch, "batch_index": batch_index}
                save_checkpoint(model, optimizer, state)

            backup_checkpoint()
            model.train()
            shard_started, running, running_steps, shard_steps = time.time(), 0.0, 0, 0
            try:
                while epoch < epochs:
                    order = np.random.default_rng(run_seed + epoch).permutation(train_starts)
                    while batch_index < batches_per_epoch:
                        step = epoch * batches_per_epoch + batch_index
                        optimizer.learning_rate = lr_at(step)
                        starts = order[batch_index * BATCH_SIZE:(batch_index + 1) * BATCH_SIZE]
                        loss, grad_norm = train_step(*make_batch(tokens, mask, starts, block_size))
                        mx.eval(loss, grad_norm, compiled_state)
                        batch_index += 1
                        steps_this_call += 1
                        shard_steps += 1
                        state["global_step"] += 1
                        state["tokens_seen"] = state.get("tokens_seen", 0) + BATCH_SIZE * block_size
                        running += loss.item()
                        running_steps += 1
                        if shard_steps % LOG_EVERY == 0:
                            rate = shard_steps * BATCH_SIZE * block_size / (time.time() - shard_started)
                            print(f"  epoch {epoch + 1}/{epochs} | step {step + 1:,}/{total_steps:,} | "
                                  f"loss {running / running_steps:.3f} | grad {grad_norm.item():.2f} | "
                                  f"lr {optimizer.learning_rate.item():.2e} | {rate:,.0f} tok/s")
                            running, running_steps = 0.0, 0
                        if shard_steps % SAVE_EVERY == 0:
                            checkpoint(done=False)
                        if MAX_STEPS is not None and steps_this_call >= MAX_STEPS:
                            checkpoint(done=False)
                            print(f"stopped after MAX_STEPS={MAX_STEPS}; run again to continue")
                            return "stopped"
                    epoch, batch_index = epoch + 1, 0
            except KeyboardInterrupt:
                checkpoint(done=False)
                print("\ninterrupted: checkpoint saved. Run All again to continue from here.")
                raise  # also stops "Run All", so the cells below do not start

            val_loss = evaluate(model, tokens, mask, val_starts, block_size)
            record = {"steps": total_steps, "tokens": int(len(tokens)), "val_loss": round(val_loss, 4),
                      "round": TRAINING_ROUND, "finished": time.strftime("%Y-%m-%d %H:%M")}
            state["trained_shards"][shard_id] = record
            state.setdefault("history", []).append({"data": shard_id, **record})
            checkpoint(done=True)
            print(f"  done: validation loss {val_loss:.3f} (perplexity {math.exp(val_loss):.1f}) | "
                  f"{(time.time() - shard_started) / 60:.1f} min | total {(time.time() - call_started) / 3600:.2f} h")
    print(f"\ntraining finished: {len(todo)} shard(s), saved {CHECKPOINT_BASE.name}")
    return "done"

## 6b. Grow a trained model into a bigger one

The first time `CHECKPOINT` (e.g. `model_002`) is used, it is created from `GROW_FROM` (`model_001_best`) at the size in
`GROW_TO_SIZE`: twice as many layers, twice as wide, twice the context. It starts out making the same predictions as
the small model (the check below shows the difference), keeps its list of trained shards, and then learns more with
its extra capacity. The optimizer starts fresh. After that, this cell does nothing.

In [7]:
def grow_model(source, target, n_layer=12, n_embd=512, block_size=512, noise=1e-3, seed=SEED):
    """Builds a bigger copy of a trained model that starts out computing the same thing, then keeps learning.

    - deeper: every layer gets twins right after it. A twin copies the layer's weights but its output
      projections are zero, so at first it passes its input through unchanged (residual connection).
    - wider: every hidden unit / attention head is duplicated and the weights reading them are divided by the
      copy count, so all outputs stay the same. A tiny bit of noise lets the copies learn different things.
    - longer context: positions beyond the old block size start as copies of the learned positions.
    The new model is saved as `target` (its own checkpoint files); `source` is not changed.
    """
    src_paths, dst_paths = checkpoint_paths(project_path(source)), checkpoint_paths(project_path(target))
    if dst_paths["weights"].exists():
        raise FileExistsError(f"{dst_paths['weights']} already exists; choose another target name")
    state = json.loads(src_paths["state"].read_text())
    old = state["model"]
    depth, width = n_layer // old["n_layer"], n_embd // old["n_embd"]
    if depth * old["n_layer"] != n_layer or width * old["n_embd"] != n_embd or depth < 1 or width < 1:
        raise ValueError("n_layer and n_embd must be whole multiples of the old model's sizes")
    if block_size < old["block_size"] or block_size % old["block_size"]:
        raise ValueError("block_size must be a multiple of the old block size")
    rng = np.random.default_rng(seed)
    w = {k: np.array(v.astype(mx.float32)) for k, v in mx.load(str(src_paths["weights"])).items()}

    def jitter(a):
        return a + rng.normal(0.0, noise * (float(a.std()) or 1.0), a.shape).astype(np.float32) if noise else a

    def square(a):        # (out, in) -> both dims duplicated, inputs averaged: same output, copied
        return jitter(np.tile(a, (width, width)) / width)

    def vector(a):        # biases, LayerNorm weights: copied
        return np.tile(a, width)

    new = {
        "token_embedding.weight": np.tile(w["token_embedding.weight"], (1, width)),
        "position_embedding.weight": np.tile(w["position_embedding.weight"], (block_size // old["block_size"], width)),
        "final_norm.weight": vector(w["final_norm.weight"]),
        "final_norm.bias": vector(w["final_norm.bias"]),
        "lm_head.weight": jitter(np.tile(w["lm_head.weight"], (1, width)) / width),
    }
    d = old["n_embd"]
    for i in range(old["n_layer"]):
        p = f"blocks.{i}."
        qkv = w[p + "attention.qkv.weight"]
        layer = {
            "attention_norm.weight": vector(w[p + "attention_norm.weight"]),
            "attention_norm.bias": vector(w[p + "attention_norm.bias"]),
            "attention.qkv.weight": np.concatenate([square(qkv[k * d:(k + 1) * d]) for k in range(3)]),
            "attention.projection.weight": square(w[p + "attention.projection.weight"]),
            "attention.projection.bias": vector(w[p + "attention.projection.bias"]),
            "feed_forward_norm.weight": vector(w[p + "feed_forward_norm.weight"]),
            "feed_forward_norm.bias": vector(w[p + "feed_forward_norm.bias"]),
            "feed_forward.up.weight": square(w[p + "feed_forward.up.weight"]),
            "feed_forward.up.bias": vector(w[p + "feed_forward.up.bias"]),
            "feed_forward.down.weight": square(w[p + "feed_forward.down.weight"]),
            "feed_forward.down.bias": vector(w[p + "feed_forward.down.bias"]),
        }
        for copy in range(depth):
            twin = dict(layer)
            if copy:  # twins start as pass-through: nothing is written back into the residual stream
                for name in ("attention.projection.weight", "attention.projection.bias",
                             "feed_forward.down.weight", "feed_forward.down.bias"):
                    twin[name] = np.zeros_like(layer[name])
            for name, value in twin.items():
                new[f"blocks.{i * depth + copy}.{name}"] = value

    shape = {"block_size": block_size, "n_embd": n_embd, "n_head": old["n_head"] * width,
             "n_layer": n_layer, "vocab_size": old["vocab_size"]}
    model = TinyGPT(shape["vocab_size"], block_size, n_embd, shape["n_head"], n_layer)
    expected = {k: v.shape for k, v in tree_flatten(model.parameters())}
    assert expected == {k: v.shape for k, v in new.items()}, "weight names/shapes do not match the model"
    model.update(tree_unflatten([(k, mx.array(v)) for k, v in new.items()]))
    mx.eval(model.parameters())

    dst_paths["weights"].parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(str(dst_paths["weights"]))
    grown_state = {**state, "model": shape, "in_progress": None,
                   "history": state.get("history", []) + [{"note": f"grown from {source}: {old['n_layer']}x{old['n_embd']} "
                              f"ctx {old['block_size']} -> {n_layer}x{n_embd} ctx {block_size}", "time": time.strftime("%Y-%m-%d %H:%M")}]}
    atomic_write_json(grown_state, dst_paths["state"])
    count = sum(v.size for v in new.values())
    print(f"grew {source} -> {target}: {n_layer} layers x {n_embd} wide, {shape['n_head']} heads, "
          f"context {block_size} | {count / 1e6:.1f}M parameters (was "
          f"{sum(v.size for v in w.values()) / 1e6:.1f}M) | optimizer starts fresh")
    return model


def compare_models(base_a, base_b, text="<user>: hello, how are you?\n<assistant>:"):
    """Max difference between two checkpoints' next-token predictions on the same short text."""
    ids = mx.array([raw_tokenizer.encode(text).ids], dtype=mx.int32)
    a, _ = load_model(base=base_a)
    b, _ = load_model(base=base_b)
    pa, pb = mx.softmax(a(ids), axis=-1), mx.softmax(b(ids), axis=-1)
    same_top = (mx.argmax(pa, axis=-1) == mx.argmax(pb, axis=-1)).mean().item()
    print(f"max probability difference {mx.abs(pa - pb).max().item():.5f} | same top token at {100 * same_top:.0f}% of positions")


if GROW_FROM and not checkpoint_paths()["weights"].exists():
    grow_model(GROW_FROM, CHECKPOINT, **GROW_TO_SIZE)
    compare_models(GROW_FROM, CHECKPOINT)  # should show (almost) the same predictions
else:
    print(f"{CHECKPOINT} already exists: nothing to grow")

checkpoints/model_002 already exists: nothing to grow


## 7. First training: the starting file in `TRAIN_FILES`

Trains the model on the one big starting file (FineWeb-Edu, about 170 shards / 12 hours for a new model). You can
interrupt any time; Run All continues where it stopped. When it is fully trained, this cell finishes in seconds.


In [8]:
train_shards = make_tokens(TRAIN_FILES)
train_model(train_shards)

making tokens for 1 files ...
  fineweb-edu-10BT-000_00000.parquet (text): 173 shards, 871.3M tokens
done: 173 shards, 871.3M tokens (0.0 min)
loaded model_002.safetensors (step 552,048, 2,458M tokens seen)
173 shards in the list: 173 already trained, 0 to train (0.0M tokens)
  fineweb-edu-10BT-000_00000.parquet: 173 of 173 shards trained

training finished: 0 shard(s), saved model_002


'done'

## 8. Chat with the model

`chat("message")` keeps the conversation history; `reset_chat()` starts over. For a back-and-forth in a loop,
run `chat_loop()` (type `exit` to stop).

In [9]:
def generate(model, prompt_ids, max_new_tokens=120, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.1,
             return_finished=False):
    ids = list(prompt_ids)
    new_ids = []
    finished = False
    for _ in range(max_new_tokens):
        context = mx.array([ids[-model.block_size:]], dtype=mx.int32)
        logits = np.array(model(context)[0, -1].astype(mx.float32))
        for token in set(new_ids[-64:]):  # discourage loops like "i am i am i am"
            logits[token] = logits[token] / repetition_penalty if logits[token] > 0 else logits[token] * repetition_penalty
        logits[[PAD_ID, UNK_ID]] = -1e9
        if temperature <= 0:
            next_id = int(logits.argmax())
        else:
            top = np.argpartition(logits, -top_k)[-top_k:]
            top = top[np.argsort(-logits[top])]
            probs = np.exp((logits[top] - logits[top].max()) / temperature)
            probs /= probs.sum()
            keep = max(1, int(np.searchsorted(np.cumsum(probs), top_p)) + 1)  # top-p: smallest set covering top_p
            next_id = int(np.random.choice(top[:keep], p=probs[:keep] / probs[:keep].sum()))
        if next_id in (EOS_ID, USER_ID, ASSISTANT_ID):
            finished = True
            break
        ids.append(next_id)
        new_ids.append(next_id)
    text = raw_tokenizer.decode(new_ids).strip()
    return (text, finished) if return_finished else text


chat_model, _ = load_model()
chat_history = []


def reset_chat():
    chat_history.clear()


def chat(message, **sampling):
    message = to_english_ascii(message) or message
    prompt = "".join(f"<user>: {u}\n<assistant>: {a}<eos>\n" for u, a in chat_history[-4:])
    prompt += f"<user>: {message}\n<assistant>:"
    prompt_ids = raw_tokenizer.encode(prompt).ids
    reply = generate(chat_model, prompt_ids[-(chat_model.block_size - 120):], **sampling)
    chat_history.append((message, reply))
    return reply


def chat_loop():
    reset_chat()
    while True:
        message = input("you: ").strip()
        if message.lower() in {"exit", "quit"}:
            break
        print("bot:", chat(message))


for question in ["hi", "hello", "how are you?", "what is a transformer?", "tell me about india.", "how are you?", "what is your name?"]:
    reset_chat()
    print(f"you: {question}\nbot: {chat(question)}\n")

loaded model_002.safetensors (step 515,467, 2,158M tokens seen)
you: hi
bot: the westernmost part of the us is called the "westernmost part of the country". it is a large area that is generally regarded as one of the most populous regions in the united states.
the westernmost part of the us is the gulf of mexico, and its smallest is the mexican continent, which is known as the mexican peninsula. it has an extensive range of territorial and geographical features, including the "germanic-buildings", the western

you: hello
bot: the kids are going to get the chance to get a lot of fun. they're just going to take the time to find something else. so if you want to get a lot of fun, it's not a great idea to start.
the kids need to go through the process and work together for them, but they can actually take a few minutes to finish. they'll get some extra money for their children. they don't have to spend a little bit of time on a regular basis. they'll get to know how to get a lot of fun whe

In [10]:
# chat_loop()

## 9. Check model accuracy

A language model has no single accuracy number, so this measures it two ways:

1. **Next-token tests on held-out text.** These use the same held-out windows that training never uses
   (every 50th window), from a fixed set per kind (shard 0 of the first wiki file and of the first parquet file in `TRAIN_FILES`, plus the txt/csv chat set), so results are comparable between checkpoints.
   - **loss / perplexity:** lower is better. Perplexity 20 means the model is about as unsure as picking between 20 equally likely tokens.
   - **top-1 accuracy:** % of next tokens the model guesses exactly. **top-5:** % where the right token is in its 5 best guesses.
   - Chat sources are scored on the assistant replies only.
2. **Chat quiz.** Fixed questions with expected keywords (greedy decoding, so results repeat exactly). Reported per
   category, plus how often the reply ends cleanly with `<eos>` and how repetitive replies are.

Each result is appended to `data/eval_history.jsonl`, and the table at the end shows progress across runs.
Evaluating while a background training run is going is fine; it only reads the checkpoint.

In [10]:
EVAL_HISTORY = project_path("data/eval_history.jsonl")

# Fixed test data, so every evaluation is comparable whatever you are training. Only held-out parts are used
# (every 50th window, which training never sees; the TinyStories validation file is never trained at all).
EVAL_FILES = {
    "wiki": "chat_dataset/enwiki-latest-pages-articles-multistream1.xml-p1p41242",
    "parquet": "chat_dataset/train-00000-of-00006-4feeb3f83346a0e9.parquet",
    "grammar": "chat_dataset/grammar/tinystories/tinystories-validation-00000-of-00001-869c898b519ad725.parquet",
}
EVAL_CHAT_TEXT_FILES = [
    "chat_dataset/chat_data.txt",
    "chat_dataset/labeled_conversations_0000.txt",
    "chat_dataset/labeled_conversations_0001.txt",
    "chat_dataset/large_english_greeting_chat_dataset.txt",
    "chat_dataset/transformer_chat_qa_dataset.txt",
]

CHAT_QUIZ = [
    # (category, question, any of these words counts as correct)
    ("small talk", "hello", ["hello", "hi", "hey"]),
    ("small talk", "hi, how are you?", ["good", "fine", "well", "great", "doing"]),
    ("small talk", "what is your name?", ["name", "i am", "i'm", "assistant", "bot"]),
    ("small talk", "thank you!", ["welcome", "glad", "pleasure", "no problem", "happy"]),
    ("small talk", "goodbye", ["bye", "see you", "take care", "goodbye"]),
    ("small talk", "can you help me?", ["help", "sure", "yes", "of course"]),
    ("knowledge", "what is the capital of france?", ["paris"]),
    ("knowledge", "what is the largest planet in the solar system?", ["jupiter"]),
    ("knowledge", "who wrote romeo and juliet?", ["shakespeare"]),
    ("knowledge", "what is water made of?", ["hydrogen", "oxygen", "h2o"]),
    ("knowledge", "tell me about albert einstein.", ["physics", "physicist", "relativity", "scientist"]),
    ("knowledge", "tell me about the moon.", ["earth", "satellite", "orbit"]),
    ("knowledge", "what is photosynthesis?", ["plant", "light", "energy", "sun"]),
    ("knowledge", "what color is the sky on a clear day?", ["blue"]),
    ("ai", "what is a transformer?", ["attention", "neural", "model", "network"]),
    ("ai", "what is machine learning?", ["data", "learn", "computer", "algorithm"]),
    ("math", "what is 2 plus 2?", ["4", "four"]),
    ("math", "what is 10 minus 3?", ["7", "seven"]),
    # made-up things: the right answer is to say it doesn't know
    ("unknown", "what is the population of the planet zorblax?", ["don't have", "don't know", "not sure", "no information", "not aware"]),
    ("unknown", "who won the 2087 football world cup?", ["don't have", "don't know", "not sure", "no information", "not aware"]),
    ("unknown", "what is the capital of the country verimonia?", ["don't have", "don't know", "not sure", "no information", "not aware"]),
]
# Reading test: SQuAD v2 validation questions (never trained). Half have the answer in the text, half do not
# (then the right reply is NO_INFO_REPLY). This measures "understands the question" and "admits missing info".
READING_TEST_FILE = "chat_dataset/understanding/squad_v2-validation.parquet"


def reading_test(model, n=60, show=4):
    path = project_path(READING_TEST_FILE)
    if not path.exists():
        return None
    rows = pq.ParquetFile(path).read(columns=["context", "question", "answers"]).to_pylist()[::7]
    answerable = [r for r in rows if r["answers"]["text"]][: n // 2]
    unanswerable = [r for r in rows if not r["answers"]["text"]][: n // 2]
    norm = lambda s: " ".join(re.findall(r"[a-z0-9]+", to_english_ascii(s).lower()))
    scores, shown = {"answerable": [], "unanswerable": []}, 0
    for group, items in (("answerable", answerable), ("unanswerable", unanswerable)):
        for row in items:
            ids = raw_tokenizer.encode(f"<user>: {to_english_ascii(reading_prompt(row))}\n<assistant>:").ids
            reply = generate(model, ids[-(model.block_size - 40):], max_new_tokens=30, temperature=0)
            if group == "answerable":
                ok = any(norm(a) and norm(a) in norm(reply) for a in row["answers"]["text"])
            else:
                ok = "don't have proper information" in reply.lower()
            scores[group].append(ok)
            if shown < show:
                expected = row["answers"]["text"][0] if row["answers"]["text"] else "(not in the text)"
                print(f"  [{'OK ' if ok else ' - '}] {row['question'][:70]}\n        -> {reply[:90]}   | expected: {expected[:40]}")
                shown += 1
    result = {group: round(100 * float(np.mean(v)), 1) for group, v in scores.items() if v}
    result["overall"] = round(float(np.mean(list(result.values()))), 1)
    return result


def eval_sets(max_windows, block_size):
    """Fixed held-out windows per test set: (name, tokens, mask, starts). Uses only tokens that already exist."""
    sets = []
    for name in ("wiki", "parquet"):
        path = project_path(EVAL_FILES[name])
        progress = load_progress(path) if path.exists() else None
        if progress and progress["shards"]:
            tokens = np.load(shard_folder(path) / progress["shards"][0]["file"], allow_pickle=False)
            sets.append((name, tokens, name == "parquet", True))
    grammar_path = project_path(EVAL_FILES["grammar"])
    if grammar_path.exists():
        texts = []
        for _, text in iter_text_parquet(grammar_path, 0):
            if text:
                texts.append(text)
            if len(texts) >= 3000:
                break
        sets.append(("grammar", np.asarray(encode_texts(texts), dtype=TOKEN_DTYPE), False, False))
    chat_paths = [project_path(f) for f in EVAL_CHAT_TEXT_FILES if project_path(f).exists()]
    if chat_paths:
        sets.append(("txt", np.asarray(encode_texts(load_txt_corpus(chat_paths)), dtype=TOKEN_DTYPE), True, True))
    result = []
    for name, tokens, chat_loss, every_50th in sets:
        mask = assistant_loss_mask(tokens) if chat_loss else np.ones(len(tokens), dtype=np.float32)
        held_out = np.arange(0, len(tokens) - block_size - 1, block_size)
        if every_50th:
            held_out = held_out[::50]  # same split as training
        result.append((name, tokens, mask, held_out[:max_windows]))
    return result


def token_metrics(model, tokens, mask, starts):
    total_loss = total_top1 = total_top5 = total_count = 0.0
    for i in range(0, len(starts), BATCH_SIZE):
        x, y, m = make_batch(tokens, mask, starts[i:i + BATCH_SIZE], model.block_size)
        logits = model(x)
        losses = nn.losses.cross_entropy(logits, y, reduction="none")
        top1 = (mx.argmax(logits, axis=-1) == y)
        top5 = (mx.argpartition(-logits, kth=4, axis=-1)[..., :5] == y[..., None]).any(axis=-1)
        total_loss += (losses * m).sum().item()
        total_top1 += (top1 * m).sum().item()
        total_top5 += (top5 * m).sum().item()
        total_count += m.sum().item()
    count = max(total_count, 1)
    loss = total_loss / count
    return {"loss": round(loss, 4), "perplexity": round(math.exp(loss), 2),
            "top1": round(100 * total_top1 / count, 2), "top5": round(100 * total_top5 / count, 2),
            "tokens": int(total_count)}


def run_chat_quiz(model, show_answers=True):
    per_category, finished_count, distinct_ratios, sentence_ok, doubled = {}, 0, [], [], []
    for category, question, keywords in CHAT_QUIZ:
        prompt_ids = raw_tokenizer.encode(f"<user>: {question}\n<assistant>:").ids
        reply, finished = generate(model, prompt_ids, max_new_tokens=60, temperature=0, return_finished=True)
        correct = any(re.search(rf"\b{re.escape(word)}\b", reply.lower()) for word in keywords)  # whole words only
        per_category.setdefault(category, []).append(correct)
        finished_count += finished
        sentence_ok.append(bool(reply) and reply.rstrip()[-1:] in ".!?")
        lowered = reply.lower().split()
        doubled.append(any(a == b for a, b in zip(lowered, lowered[1:])))
        words = reply.split()
        pairs = list(zip(words, words[1:]))
        distinct_ratios.append(len(set(pairs)) / len(pairs) if pairs else 1.0)
        if show_answers:
            print(f"  [{'OK ' if correct else ' - '}] {question}\n        -> {reply[:150]}")
    scores = {category: round(100 * sum(v) / len(v), 1) for category, v in per_category.items()}
    all_answers = [x for v in per_category.values() for x in v]
    scores["overall"] = round(100 * sum(all_answers) / len(all_answers), 1)
    return {"keyword_score": scores,
            "ends_with_eos": round(100 * finished_count / len(CHAT_QUIZ), 1),
            "non_repetitive": round(100 * float(np.mean(distinct_ratios)), 1),
            "complete_sentences": round(100 * float(np.mean(sentence_ok)), 1),
            "doubled_words": round(100 * float(np.mean(doubled)), 1)}


def evaluate_model(checkpoint=None, max_windows=200, show_answers=True, save=True):
    """Scores a checkpoint (default: CHECKPOINT). Example: evaluate_model("checkpoints/chatbot_mlx.previous")."""
    model, state = load_model(base=checkpoint)
    started = time.time()
    print("preparing held-out data:")
    held_out_sets = eval_sets(max_windows, model.block_size)
    print("\nnext-token tests on held-out text:")
    print(f"  {'source':<8} {'loss':>7} {'perplexity':>11} {'top-1 %':>8} {'top-5 %':>8} {'tokens':>9}")
    metrics = {}
    for name, tokens, mask, starts in held_out_sets:
        metrics[name] = token_metrics(model, tokens, mask, starts)
        r = metrics[name]
        print(f"  {name:<8} {r['loss']:>7.3f} {r['perplexity']:>11.1f} {r['top1']:>8.1f} {r['top5']:>8.1f} {r['tokens']:>9,}")
    print("\nchat quiz:")
    quiz = run_chat_quiz(model, show_answers)
    print("  keyword score % :", quiz["keyword_score"])
    print(f"  ends with <eos> : {quiz['ends_with_eos']}%   non-repetitive: {quiz['non_repetitive']}%")
    print(f"  grammar checks  : {quiz['complete_sentences']}% end as a full sentence, "
          f"{quiz['doubled_words']}% have a doubled word like 'the the'")
    print("\nreading test (answer from a text, or say the information is not there):")
    reading = reading_test(model, show=4 if show_answers else 0)
    if reading:
        print(f"  answerable correct: {reading['answerable']}%   unanswerable -> says no info: "
              f"{reading['unanswerable']}%   overall: {reading['overall']}%")
    print(f"  ({time.time() - started:.0f}s)")
    result = {"reading": reading or {}, "time": time.strftime("%Y-%m-%d %H:%M"), "checkpoint": str(checkpoint or CHECKPOINT),
              "global_step": state["global_step"], "tokens_seen": state.get("tokens_seen", 0),
              "trained_shards": len(state.get("trained_shards", {})), "metrics": metrics, **quiz}
    if save:
        EVAL_HISTORY.parent.mkdir(parents=True, exist_ok=True)
        with EVAL_HISTORY.open("a") as handle:
            handle.write(json.dumps(result) + "\n")
    return result


def show_eval_history(last=15):
    if not EVAL_HISTORY.exists():
        print("no evaluations yet")
        return
    rows = [json.loads(line) for line in EVAL_HISTORY.read_text().splitlines() if line.strip()][-last:]
    print(f"\n{'time':<17} {'step':>9} {'shards':>6} | {'wiki ppl':>8} {'wiki top1':>9} | "
          f"{'chat ppl':>8} {'chat top1':>9} | {'gram ppl':>8} | {'quiz %':>6} {'eos %':>6} {'read %':>6}")
    for row in rows:
        wiki = row["metrics"].get("wiki", {})
        chat_metrics = row["metrics"].get("parquet") or row["metrics"].get("txt") or {}
        print(f"{row['time']:<17} {row['global_step']:>9,} {row['trained_shards']:>6} | "
              f"{wiki.get('perplexity', float('nan')):>8.1f} {wiki.get('top1', float('nan')):>9.1f} | "
              f"{chat_metrics.get('perplexity', float('nan')):>8.1f} {chat_metrics.get('top1', float('nan')):>9.1f} | "
              f"{row['metrics'].get('grammar', {}).get('perplexity', float('nan')):>8.1f} | "
              f"{row['keyword_score']['overall']:>6.1f} {row['ends_with_eos']:>6.1f} {row.get('reading', {}).get('overall', float('nan')):>6.1f}")


evaluation = evaluate_model()
show_eval_history()

loaded model_002.safetensors (step 552,048, 2,458M tokens seen)
preparing held-out data:
  chat_data.txt: 20 conversations, 20 kept
  labeled_conversations_0000.txt: 2,091 conversations, 1,921 kept
  labeled_conversations_0001.txt: 22 conversations, 21 kept
  large_english_greeting_chat_dataset.txt: 9,500 conversations, 2,620 kept
  transformer_chat_qa_dataset.txt: 20 conversations, 1 kept

next-token tests on held-out text:
  source      loss  perplexity  top-1 %  top-5 %    tokens
  wiki       3.337        28.1     39.4     59.7   101,376
  parquet    2.826        16.9     46.6     68.6    78,273
  grammar    3.080        21.8     38.2     62.1   102,400
  txt        3.859        47.4     31.6     53.4     1,886

chat quiz:


NameError: name 'generate' is not defined

In [ ]:
# Compare with the checkpoint from before the last training run:
# evaluate_model("checkpoints/chatbot_mlx.previous", save=False)

## 10. Retrain: one file per run, in list order

Uses the **same functions**: `next_files_to_train(...)` picks the first entry in `RETRAIN_FILES` that is not fully
trained yet (one file, or a `( ... )` group of files trained together, e.g. a wiki file with a chat file), `make_tokens(...)` tokenizes it, and `train_model(...)` continues training the same checkpoint on it.
Then accuracy is checked again. Run this cell (or Run All) again for the next file. The order alternates knowledge
and chat and ends with chat, so the model answers like a chatbot at the end. Set `ONE_FILE_PER_RUN = False` to
train the whole list in one run.


In [ ]:
RETRAIN_FILES = [
    # One entry per run, top to bottom (ONE_FILE_PER_RUN = True). An entry is one file, or a ( ... ) group of files
    # trained together in one run, interleaved (MIX_FILES = True): e.g. a wiki file with a chat file, so neither one
    # takes over the model's style. Finished entries are skipped automatically; you can also comment lines out.

    # --- already trained by model_001 (kept so a fresh model gets them too)
    "chat_dataset/grammar/simplewiki/simplewiki-latest-pages-articles.xml",
    "chat_dataset/train-00000-of-00006-4feeb3f83346a0e9.parquet",
    "chat_dataset/enwiki-latest-pages-articles-multistream1.xml-p1p41242",
    # SKIP: smol-rewrite is a text-rewriting task, not chat (it taught model_001 to answer every message with an email)
    # "chat_dataset/grammar/smoltalk/smoltalk-smol-rewrite.parquet",

    # --- next: plain chat first, to undo the email habit
    "chat_dataset/train-00002-of-00006-1779b7cec9462180.parquet",

    # --- then knowledge + chat in pairs
    ("chat_dataset/enwiki-latest-pages-articles-multistream2.xml-p41243p151573", "chat_dataset/train-00001-of-00006-4030672591c2f478.parquet"),
    ("chat_dataset/enwiki-latest-pages-articles-multistream3.xml-p151574p311329", "chat_dataset/train-00003-of-00006-2fa862bfed56af1f.parquet"),
    ("chat_dataset/enwiki-latest-pages-articles-multistream4.xml-p311330p558391", "chat_dataset/train-00004-of-00006-18f4bdd50c103e71.parquet"),
    ("chat_dataset/enwiki-latest-pages-articles-multistream5.xml-p558392p958045", "chat_dataset/train-00005-of-00006-fe1acc5d10a9f0e2.parquet"),
    # the big high-quality chat set, spread through two wiki parts
    # NOT NEEDED for model_001: wiki stopped improving after wiki 3-4, and the wiki-heavy pairs made the
    # "1996 american comedy film" habit stronger. Kept for a bigger (grown) model later.
    # ("chat_dataset/enwiki-latest-pages-articles-multistream6.xml-p958046p1483661", "chat_dataset/enwiki-latest-pages-articles-multistream7.xml-p1483662p2134111", "chat_dataset/grammar/smoltalk/smoltalk-magpie-ultra-00000.parquet"),
    # instead: the big high-quality chat set on its own, to bring back normal chat replies
    "chat_dataset/grammar/smoltalk/smoltalk-magpie-ultra-00000.parquet",
    # optional extra knowledge (more wiki = more article style; keep a chat entry after it)
    # ("chat_dataset/enwiki-latest-pages-articles-multistream8.xml-p2134112p2936260", "chat_dataset/enwiki-latest-pages-articles-multistream9.xml-p2936261p4045402"),

    # --- final polish of model_001 (replaced for model_002 by the polish entry below, which adds OpenAssistant)
    # ("chat_dataset/grammar/smoltalk/smoltalk-everyday-conversations.parquet",
    #  "chat_dataset/chat_data.txt",
    #  "chat_dataset/labeled_conversations_0000.txt",
    #  "chat_dataset/labeled_conversations_0001.txt",
    #  "chat_dataset/large_english_greeting_chat_dataset.txt",
    #  "chat_dataset/transformer_chat_qa_dataset.txt"),

    # ===== model_002 (grown, 51M parameters, 512-token context): understanding + knowledge =====
    # Everything above is already trained (model_002 keeps model_001_best's record), so training starts here.
    # 1) knowledge (FineWeb-Edu) mixed with reading questions (SQuAD v2: answer from the text, or reply
    #    "i don't have proper information") and human-written Q&A (Dolly). ~0.9B tokens, about 2.5-3 days.
    ("chat_dataset/grammar/fineweb_edu/fineweb-edu-10BT-001_00000.parquet", "chat_dataset/understanding/squad_v2-train.parquet", "chat_dataset/understanding/databricks-dolly-15k.jsonl"),
    # 2) more knowledge: FineWeb-Edu + wiki parts the model has not seen. ~1.1B tokens, about 3-3.5 days.
    #    Comment out to skip; the polish entry below still runs.
    ("chat_dataset/grammar/fineweb_edu/fineweb-edu-10BT-002_00000.parquet", "chat_dataset/enwiki-latest-pages-articles-multistream6.xml-p958046p1483661", "chat_dataset/enwiki-latest-pages-articles-multistream7.xml-p1483662p2134111"),
    # 3) final polish, always last: human-written conversations (OpenAssistant, English only), small talk, your txt files
    ("chat_dataset/understanding/oasst_top1_2023-08-25_train.jsonl",
     "chat_dataset/grammar/smoltalk/smoltalk-everyday-conversations.parquet",
     "chat_dataset/chat_data.txt",
     "chat_dataset/labeled_conversations_0000.txt",
     "chat_dataset/labeled_conversations_0001.txt",
     "chat_dataset/large_english_greeting_chat_dataset.txt",
     "chat_dataset/transformer_chat_qa_dataset.txt"),

    # optional: TinyStories (simple grammar). Only for a FRESH model, early in the list, in a group with a chat file.
    # "chat_dataset/grammar/tinystories/tinystories-train-00000-of-00004-2d5a1467fff1081b.parquet",
]

files_now = next_files_to_train(RETRAIN_FILES) if ONE_FILE_PER_RUN else flatten_files(RETRAIN_FILES)
if files_now:
    retrain_shards = make_tokens(files_now)
    train_model(retrain_shards)

# Reload the newly trained weights, then check accuracy again (the history table shows before vs after).
chat_model, _ = load_model()
evaluation = evaluate_model()
show_eval_history()

entry 10 of 12 in the list (9 done): fineweb-edu-10BT-001_00000.parquet, squad_v2-train.parquet, databricks-dolly-15k.jsonl
making tokens for 3 files ...
  fineweb-edu-10BT-001_00000.parquet (text): 173 shards, 869.5M tokens
  squad_v2-train.parquet (parquet): 7 shards, 30.3M tokens
  databricks-dolly-15k.jsonl (jsonl): 1 shards, 3.2M tokens
done: 181 shards, 903.0M tokens (0.0 min)
training order: shards of all files interleaved (MIX_FILES = True)
loaded model_002.safetensors (step 552,048, 2,458M tokens seen)
181 shards in the list: 79 already trained, 102 to train (504.9M tokens)
  fineweb-edu-10BT-001_00000.parquet: 76 of 173 shards trained
  squad_v2-train.parquet: 3 of 7 shards trained
  databricks-dolly-15k.jsonl: 0 of 1 shards trained
  continuing fineweb-edu-10BT-001_00000.parquet/shard_000076.npy at batch 346 (saved position)

=== this run 1/102 | fineweb-edu-10BT-001_00000.parquet shard 76 (76 of 173 done) (text) | 5,033,540 tokens | 602 steps | lr 0.0001 | loss on all token

## 11. Notes for later runs

| You want to... | Do this |
|---|---|
| Continue after stopping | Run All again. Finished shards are skipped, an interrupted shard resumes. |
| Train the next file | Run All (or the Retrain cell) again: it picks the next unfinished file of `RETRAIN_FILES`. |
| Train on new files | Add them to `RETRAIN_FILES` (or `TRAIN_FILES`) and run. Only new shards are trained. |
| Train over everything again | Set `TRAINING_ROUND = 2` (then 3, 4 ...). |
| Undo the last shard | Copy `checkpoints/chatbot_mlx.previous.*` over `checkpoints/chatbot_mlx.*`. |
| Start a brand-new model | Set `CHECKPOINT = "checkpoints/chatbot_fresh"` (any new name; optionally change `NEW_MODEL`) and Run All. The old model stays untouched. |

Changing a txt/csv file makes a new chat set that is trained again. Delete a folder in `data/shards/` only if you change
the text-cleaning code or the tokenizer.

In [75]:
for _ in range(7):
    reset_chat()
    print(chat("good morning"))

hi! i am ready when you are.
hi! ask me anything you need help with.
hi there! let me know what you need.
hi! i am ready when you are.
hello! what can i help you with?
hi! tell me what you need help with.
hi there! let me know what you need.


In [31]:
chat('how are you doing?')

"i'm doing well, thank you. how can i help you?"

In [30]:
chat('tell me what is united stats of america?')

"united stats of america's fantasy world is a genre of art that goes into its own way."